In [ ]:
'''
The dataset comprises SDO/HMI continuum FITS files spanning from June 3, 2012 (13:30:00 TAI) to June 6, 2012 (13:28:30 TAI), providing a continuous 45-second cadence for modeling the Venusian transit.
The code runs successfully on Colab, the above files were downloaded from jsoc at an average cadence of 2 minutes and stored in drive folder 'VenusTransit2012'
'''
!pip install -q batman-package emcee corner h5py

import os
import glob
import gc
import shutil
import time
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.time import Time
import batman
import emcee
from emcee.backends import HDFBackend
import corner
from google.colab import drive

# =========================================================
# 1. SETUP & MOUNT (RATE-LIMIT BYPASS)
# =========================================================
print("[1/5] Mounting Drive and Copying to local SSD...")
drive.mount('/content/drive', force_remount=True)
os.makedirs("/content/VenusData", exist_ok=True)

drive_path = '/content/drive/MyDrive/VenusTransit2012/*.fits'
source_files = glob.glob(drive_path)

if not source_files:
    raise ValueError("Google Drive mount is empty. Restart Colab runtime and try again.")

print(f"Found {len(source_files)} files. Copying safely to prevent Drive disconnect...")

for i, f in enumerate(source_files):
    dest = os.path.join("/content/VenusData", os.path.basename(f))
    if not os.path.exists(dest):
        try:
            shutil.copy2(f, dest)
        except OSError:
            time.sleep(2)
            shutil.copy2(f, dest)

    if (i + 1) % 300 == 0:
        print(f"Copied {i + 1} / {len(source_files)}...")

all_files = sorted(glob.glob('/content/VenusData/*.fits'))
files_ref = [f for f in all_files if '2012.06.03' in f or '2012.06.04' in f]
files_trans = [f for f in all_files if '2012.06.05' in f or '2012.06.06' in f]

if not files_ref or not files_trans:
    raise ValueError("Copy finished, but the date strings were not found in the filenames.")
print(f"SUCCESS: {len(files_ref)} Reference files and {len(files_trans)} Transit files ready.")


# =========================================================
# 2. MEMORY-SAFE EXTRACTION & DETRENDING
# =========================================================
print("\n[2/5] Extracting Photometry (Memory-Safe)...")

def get_flux_safe(files):
    times, fluxes = [], []
    for i, f in enumerate(files):
        try:
            with fits.open(f, memmap=True) as hdul:
                idx = 1 if len(hdul) > 1 else 0
                header = hdul[idx].header
                exptime = max(header.get('EXPTIME', 1.0), 1.0)
                if hdul[idx].data is not None:
                    fluxes.append(np.nansum(hdul[idx].data) / exptime)
                    t_str = header.get("DATE-OBS", header.get("T_OBS"))
                    times.append(Time(t_str).datetime)
                hdul.close()
        except Exception: continue
        if (i + 1) % 500 == 0: gc.collect()
    return np.array(times), np.array(fluxes)

t_ref_raw, f_ref_raw = get_flux_safe(files_ref)
t_trans_raw, f_trans_raw = get_flux_safe(files_trans)

print("[3/5] Detrending Data (Wobble & Polish)...")
t_ref_h = np.array([(t - t_ref_raw[0]).total_seconds()/3600.0 for t in t_ref_raw])
t_trans_h = np.array([(t - t_trans_raw[0]).total_seconds()/3600.0 for t in t_trans_raw])

f_ref_norm = f_ref_raw / np.median(f_ref_raw)
f_trans_norm = f_trans_raw / np.median(f_trans_raw)

coeffs_wobble = np.polyfit(t_ref_h, f_ref_norm, 16)
detrended_flux = f_trans_norm / np.polyval(coeffs_wobble, t_trans_h)

t_mid = Time("2012-06-06T01:29:35").datetime
t_days_from_mid = np.array([(t - t_mid).total_seconds()/86400.0 for t in t_trans_raw])

mask_wings = (t_days_from_mid < -0.16) | (t_days_from_mid > 0.16)
polish_coeffs = np.polyfit(t_days_from_mid[mask_wings], detrended_flux[mask_wings], 2)
final_flux = detrended_flux / np.polyval(polish_coeffs, t_days_from_mid)
final_flux /= np.median(final_flux[mask_wings])




In [ ]:
# =========================================================
# 3. BAYESIAN MCMC SETUP (WITH DRIVE SAVE-STATE FIX)
# =========================================================
print("\n[4/5] Running MCMC Optimizer with Save-State...")
import os

flux_err = np.std(final_flux[mask_wings])
mask_fit = (t_days_from_mid > -0.22) & (t_days_from_mid < 0.22)
t_fit = t_days_from_mid[mask_fit]
f_fit = final_flux[mask_fit]

def log_prior(theta):
    rp, t0, b, a_rs = theta
    if (0.02 < rp < 0.04 and -0.01 < t0 < 0.01 and 0.0 <= b < 1.0 and 150.0 < a_rs < 250.0):
        return 0.0
    return -np.inf

def log_likelihood(theta):
    rp, t0, b, a_rs = theta
    inc = np.degrees(np.arccos(b / a_rs))
    params = batman.TransitParams()
    params.t0, params.per, params.rp, params.a, params.inc = t0, 224.7, rp, a_rs, inc
    params.ecc, params.w = 0., 90.
    params.u, params.limb_dark = [0.616, -0.252, 0.825, -0.429], "nonlinear"
    sim_flux = batman.TransitModel(params, t_fit).light_curve(params)
    return -0.5 * np.sum(((f_fit - sim_flux) / flux_err) ** 2)

def log_probability(theta):
    lp = log_prior(theta)
    if not np.isfinite(lp): return -np.inf
    return lp + log_likelihood(theta)

ndim, nwalkers, max_steps = 4, 32, 2500
backend_filename = "/content/drive/MyDrive/venus_mcmc_save.h5"
backend = HDFBackend(backend_filename)

# FIX: Check if the file actually exists before asking for the iteration count
if os.path.exists(backend_filename) and backend.iteration > 0:
    print(f"Detected existing save. Resuming from step {backend.iteration}...")
    pos = None
else:
    print("Starting fresh MCMC run...")
    backend.reset(nwalkers, ndim)
    initial_guess = np.array([0.0298, 0.0, 0.586, 215.0])
    pos = initial_guess + 1e-4 * np.random.randn(nwalkers, ndim)

sampler = emcee.EnsembleSampler(nwalkers, ndim, log_probability, backend=backend)

if os.path.exists(backend_filename) and backend.iteration >= max_steps:
    print("MCMC already completed 2500 steps. Moving to results.")
else:
    # If resuming, pos will be None and it picks up automatically
    sampler.run_mcmc(pos, max_steps - (backend.iteration if os.path.exists(backend_filename) else 0), progress=True)

In [ ]:
# =========================================================
# 4. RIGOROUS RESULTS EXTRACTION
# =========================================================
print("Extracting samples...")
# This defines the 'samples' variable your plot is looking for
samples = sampler.get_chain(discard=500, flat=True)

# Calculate medians (res) and errors (err)
res = np.median(samples, axis=0)
err = np.std(samples, axis=0)

# Unpack for the plotting script
best_rp, best_t0, best_b, best_a_rs = res

print("Samples extracted. You can now run your visualization block.")

In [ ]:
# =========================================================
# 5. VISUALIZATION (1-SIGMA SHADED FIT + CORNER PLOT)
# =========================================================
# Rebuild the theoretical curve
params = batman.TransitParams()
params.t0, params.per, params.rp, params.a = best_t0, 224.7, best_rp, best_a_rs
params.inc = np.degrees(np.arccos(best_b / best_a_rs))
params.ecc, params.w = 0., 90.
params.u, params.limb_dark = [0.616, -0.252, 0.825, -0.429], "nonlinear"
best_fit_curve = batman.TransitModel(params, t_fit).light_curve(params)

# Generate 100 random curves from the chain to show error visually
inds = np.random.randint(len(samples), size=100)
flux_samples = np.empty((100, len(t_fit)))
for i, ind in enumerate(inds):
    rp, t0, b, a_rs = samples[ind]
    params.rp, params.t0, params.b, params.a = rp, t0, b, a_rs
    params.inc = np.degrees(np.arccos(b / a_rs))
    flux_samples[i] = batman.TransitModel(params, t_fit).light_curve(params)

lower_1sigma = np.percentile(flux_samples, 16, axis=0)
upper_1sigma = np.percentile(flux_samples, 84, axis=0)

# Plotting the Transit Fit
fig, (ax1, ax2) = plt.subplots(2, 1, gridspec_kw={'height_ratios': [3, 1]}, figsize=(10, 8), sharex=True)

# Main Plot
ax1.scatter(t_fit, f_fit, color='black', s=5, alpha=0.3, label='SDO/HMI Data')
ax1.fill_between(t_fit, lower_1sigma, upper_1sigma, color='yellow', alpha=0.6, label='1-Sigma Uncertainty')
ax1.plot(t_fit, best_fit_curve, color='red', linewidth=1.5, label='Median Best Fit')
ax1.set_ylabel("Normalized Flux")
ax1.set_title("Venus Transit 2012: Data vs. MCMC Model")
ax1.legend(loc="lower right")
ax1.grid(True, linestyle='--', alpha=0.5)


# Calculate actual depth from the lowest point of the limb-darkened curve
actual_depth = 1.0 - np.min(best_fit_curve)
depth_ppm = actual_depth * 1e6

# Print statement for accurate depth
print(f"Calculated Transit Depth: {depth_ppm:.3f} ppm")

# Text Box (Matches your exact printed precision)
textstr = '\n'.join((
    f'Transit Depth: {depth_ppm:.3f} ppm',
    f'$R_p/R_s$: {best_rp:.5f}',
    f'Impact ($b$): {best_b:.4f}',
    f'$a/R_s$: {best_a_rs:.5f}'
))
props = dict(boxstyle='square', facecolor='white', alpha=1.0, edgecolor='black')
ax1.text(0.03, 0.15, textstr, transform=ax1.transAxes, fontsize=11,
        verticalalignment='bottom', bbox=props)

# Residuals Plot
residuals = f_fit - best_fit_curve
ax2.scatter(t_fit, residuals, color='gray', s=5, alpha=0.4)
ax2.axhline(0, color='red', linestyle='--')
ax2.set_xlabel("Time from Mid-Transit (Days)")
ax2.set_ylabel("Residuals")
ax2.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

# Plotting the Corner Plot
fig2 = corner.corner(
    samples,
    labels=["$R_p/R_s$", "$t_0$", "$b$", "$a/R_s$"],
    truths=res,
    show_titles=True,
    title_fmt=".5f"
)
plt.show()

In [ ]:
# EXACT instantaneous distances for June 5-6, 2012
venus_earth = 0.28870298598567  # AU
sun_earth = 1.01473918072931    # AU

a_venus = sun_earth - venus_earth
d_earth = sun_earth
R_SUN_KM = 695700.0

correction_factor = 1.0 - (a_venus / d_earth)

# Calculate final corrected radius using the ENTIRE chain
radius_chain_km = samples[:, 0] * R_SUN_KM * correction_factor
calc_radius = np.median(radius_chain_km)
calc_radius_err = np.std(radius_chain_km)

print(f"\n==========================================")
print(f"📊 FINAL MCMC ESTIMATES 📊")
print(f"==========================================")
print(f"Radius Ratio (Rp/Rs):       {res[0]:.5f} ± {err[0]:.5f}")
print(f"Mid-Transit Time (t0):      {res[1]:.5f} ± {err[1]:.5f} days")
print(f"Impact Parameter (b):       {res[2]:.4f} ± {err[2]:.4f}")
print(f"Scaled Semi-major (a/Rs):   {res[3]:.5f} ± {err[3]:.5f}")
print(f"------------------------------------------")
print(f"Calculated Radius:          {calc_radius:,.0f} km ± {calc_radius_err:,.0f} km")
print(f"True Optical Radius:        6131 km")
print(f"Accuracy:                   {100 - (abs(6131-calc_radius)/6131)*100:.2f}%")
print(f"==========================================")